# Model Comparison Report

This notebook loads the saved artifacts from the three experiment notebooks and builds summary tables and plots for the shared evaluation metrics.

In [ ]:
!pip install -q pandas matplotlib seaborn

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns


sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 6)

In [ ]:
EXPERIMENTS = {
    "baseline_bpe": "Transformer + BPE",
    "hierarchical_whitespace": "Hierarchical + Whitespace",
    "learned_segmentation": "Hierarchical + Learned Segmentation",
}


def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


rows = []
artifacts = {}
for key, label in EXPERIMENTS.items():
    metrics_path = Path(f"{key}_metrics.json")
    config_path = Path(f"{key}_config.json")
    sample_path = Path(f"{key}_sample.txt")

    metrics = load_json(metrics_path) if metrics_path.exists() else {}
    config = load_json(config_path) if config_path.exists() else {}
    sample = sample_path.read_text(encoding="utf-8") if sample_path.exists() else ""

    artifacts[key] = {
        "label": label,
        "metrics": metrics,
        "config": config,
        "sample": sample,
    }

    rows.append(
        {
            "model_key": key,
            "model": label,
            "bpb": metrics.get("bpb"),
            "noisy_bpb": metrics.get("noisy_bpb"),
            "ood_bpb": metrics.get("ood_bpb"),
            "boundary_f1": (metrics.get("boundary_f1") or {}).get("f1"),
            "boundary_precision": (metrics.get("boundary_f1") or {}).get("precision"),
            "boundary_recall": (metrics.get("boundary_f1") or {}).get("recall"),
        }
    )

results_df = pd.DataFrame(rows)
results_df

## Summary Table

In [ ]:
summary_df = results_df.copy()
for column in ["bpb", "noisy_bpb", "ood_bpb", "boundary_f1", "boundary_precision", "boundary_recall"]:
    summary_df[column] = summary_df[column].map(lambda x: round(x, 4) if pd.notna(x) else None)
summary_df

## Main Metrics

In [ ]:
plot_df = results_df.melt(
    id_vars=["model"],
    value_vars=["bpb", "noisy_bpb", "ood_bpb"],
    var_name="metric",
    value_name="value",
).dropna()

metric_labels = {
    "bpb": "In-domain BPB",
    "noisy_bpb": "Noisy BPB",
    "ood_bpb": "OOD BPB",
}
plot_df["metric"] = plot_df["metric"].map(metric_labels)

ax = sns.barplot(data=plot_df, x="metric", y="value", hue="model", palette="deep")
ax.set_title("Byte-Normalized Language Modeling Metrics")
ax.set_xlabel("")
ax.set_ylabel("Bits per byte")
ax.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
boundary_df = results_df.dropna(subset=["boundary_f1"])

if not boundary_df.empty:
    ax = sns.barplot(data=boundary_df, x="model", y="boundary_f1", palette="crest")
    ax.set_title("Boundary F1")
    ax.set_xlabel("")
    ax.set_ylabel("F1")
    ax.set_ylim(0, 1.05)
    plt.xticks(rotation=10)
    plt.tight_layout()
    plt.show()
else:
    print("Boundary metrics are not available yet.")

## Sample Generations

In [ ]:
for key, artifact in artifacts.items():
    print(f"\n=== {artifact['label']} ===\n")
    print(artifact["sample"][:1200] if artifact["sample"] else "No sample saved.")

## Paper-Ready Table

In [ ]:
paper_df = summary_df[["model", "bpb", "noisy_bpb", "ood_bpb", "boundary_f1"]].copy()
paper_df.columns = ["Model", "BPB", "Noisy BPB", "OOD BPB", "Boundary F1"]
paper_df

In [ ]:
paper_df.to_csv("comparison_summary.csv", index=False)
Path("comparison_table.tex").write_text(paper_df.to_latex(index=False, na_rep="-"), encoding="utf-8")
print("Saved comparison_summary.csv and comparison_table.tex")